In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv("../data/Telco_Customer-Churn.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

- customerID : 고객의 고유 식별자
- gender : 성별
- SeniorCitizen : 고령자 여부 (1 : 고령자, 0 : 고령자가 아님)
- Partner : 결혼 여부 또는 파트너 여부 (Yes / No)
- Dependents : 부양 가족 여부 (Yes / No)
- tenure : 고객이 서비스를 이용한 기간 (개월 수)
- PhoneService : 전화 서비스 가입 여부 ( Yes / No )
- MultipleLines : 다중 회선(전화선 2개 이상) 가입 여부 (Yes / No / No phone service)
- InternetService : 인터넷 서비스의 제공 방식 (DSL, Fiber, optic(광랜), No)
- OnlineSecurity : 온라인 보안 서비스 가입 여부 ( Yes / No / No internet service )
- OnlineBackup : 온라인 백업 서비스 가입 여부 ( Yes / No / No internet service )
- DeviceProtection : 기기 보호 서비스 가입 여부 ( Yes / No / No internet service )
- TechSupport : 기술 지원 서비스 가입 여부 ( Yes / No / No internet service )
- StreamingTV : 스트리밍 TV 서비스 가입 여부 ( Yes / No / No internet service )
- StreamingMovies : 스트리밍 영화 서비스 가입 여부 ( Yes / No / No internet service )
- Contract : 계약 유형 ( Month-to-month : 월정액,  One year : 1년 계약, Two year : 2년 계약 )
- PaperlessBilling : 종이 없는 청구서(이메일, 모바일 등 전자 청구서) 사용 여부 ( Yes / No )
- PaymentMethod : 요금 결제 방식 (Electronic check, Mailed check, Bank transfer (automatic), Credit card (automatic))
- MonthlyCharges : 매월 고객에게 청구되는 요금 
- TotalCharges : 고객이 가입한 이후 총 청구된 누적 요금
- Churn : 고객의 해지 여부 ( Yes : 해지, No : 유지 )

In [ ]:
df.loc[df['TotalCharges'] == ' ', ]

In [8]:
# 문자형 데이터를 숫자형 변환 -> astype() : 변환시 문제가 발생하면 에러!
# to_numeric() -> 에러 발생시 결측치로 대체 
# errors 매개변수 
#   'raise'(기본값) : 변환시 문제가 발생하면 에러 
#   'coerce' : 변환이 되지 않는 데이터는 결측치 처리 
#   'ignore' : 변환 중 문제가 발생하면 기존 데이터로 그래도 반환
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors = 'coerce')

In [11]:
# TotalCharges가 결측치인 데이터는 신규 가입자 
# df['TotalCharges'].isna().sum()
# 신규 가입자는 이탈 여부와는 관계가 없는 데이터셋으로 간주하고 제거 
df.dropna(inplace=True)

In [30]:
# 더미화
df2 = pd.get_dummies(df, columns=['InternetService', 'PaymentMethod', 'Contract'])

In [31]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 28 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   customerID                               7032 non-null   object 
 1   gender                                   7032 non-null   object 
 2   SeniorCitizen                            7032 non-null   int64  
 3   Partner                                  7032 non-null   object 
 4   Dependents                               7032 non-null   object 
 5   tenure                                   7032 non-null   int64  
 6   PhoneService                             7032 non-null   object 
 7   MultipleLines                            7032 non-null   object 
 8   OnlineSecurity                           7032 non-null   object 
 9   OnlineBackup                             7032 non-null   object 
 10  DeviceProtection                         7032 non-nul

In [32]:
df2.drop('customerID', axis=1, inplace=True)

In [33]:
le = LabelEncoder()
df2['gender'] = le.fit_transform(df2['gender'])

In [34]:
# 나머지 Object 형 컬럼들을 Yes는 1로 변환, 나머지는 0으로 변환 
df2['OnlineSecurity'].map(
    lambda x : 1 if x == 'Yes' else 0
).unique()

array([0, 1])

In [35]:
obj_cols = df2.select_dtypes('object').columns
obj_cols

Index(['Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'Churn'],
      dtype='object')

In [36]:
for col in obj_cols:
    df2[col] = df2[col].map(
        lambda x : 1 if x == 'Yes' else 0
    )

In [ ]:
df2.info()

In [39]:
df2['Churn'].value_counts()

Churn
0    5163
1    1869
Name: count, dtype: int64

In [40]:
x = df2.drop('Churn', axis=1)
y = df2['Churn']

In [41]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

In [42]:
model = RandomForestClassifier(random_state=42, class_weight="balanced_subsample")

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [43]:
from sklearn.metrics import classification_report

In [44]:
pred = model.predict(X_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1033
           1       0.62      0.49      0.55       374

    accuracy                           0.78      1407
   macro avg       0.72      0.69      0.70      1407
weighted avg       0.77      0.78      0.78      1407



In [45]:
# 인사이트 도출 : 어떤 변수가 이탈에 가장 큰 영향을 주는것인가?
# 피쳐의 중요도 
feature_df = pd.DataFrame(
    {
        'feature_name' : x.columns, 
        'importance' : model.feature_importances_
    }
).sort_values('importance', ascending=False).reset_index(drop=True)

In [46]:
print('해적 지표 Retention 분석 : 이탈에 영향을 미치는 핵심 요인 Top 5')
print(feature_df.head())

해적 지표 Retention 분석 : 이탈에 영향을 미치는 핵심 요인 Top 5
                  feature_name  importance
0                 TotalCharges    0.160220
1                       tenure    0.160197
2               MonthlyCharges    0.153861
3      Contract_Month-to-month    0.098481
4  InternetService_Fiber optic    0.038488
